# IDEA-001: Supply in Profit Entry Signal

**Hypothesis:** When Supply in Profit < 50%, more than half of all BTC is underwater = extreme fear = buy.

**Logic:**
- Supply in Profit = % of all BTC that was last moved at a price BELOW current price
- When < 50%, majority of holders are at a loss
- Historically rare - only happens in deep bear markets / capitulation
- Similar to SOPR < 1 but measures stock, not flow

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("Supply in Profit Exploration 🔍")

In [ ]:
# Load data
DATA_DIR = Path("../data/daily")

sip = pd.read_parquet(DATA_DIR / "supply_in_profit_percent.parquet").rename(columns={"value": "sip_decimal"}).set_index("time")
price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")

df = sip.join(price, how='inner').join(mvrv, how='inner').join(sopr, how='inner').join(sopr_sth, how='inner')
df = df.sort_index()

# Convert decimal to percentage (0.7 -> 70%)
df['sip_pct'] = df['sip_decimal'] * 100

print(f"Data: {len(df)} rows")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")

---
## 1. Understand the Data

In [ ]:
# Basic stats
print("SUPPLY IN PROFIT % STATISTICS")
print("="*50)
print(f"Min: {df['sip_pct'].min():.1f}%")
print(f"Max: {df['sip_pct'].max():.1f}%")
print(f"Mean: {df['sip_pct'].mean():.1f}%")
print(f"Median: {df['sip_pct'].median():.1f}%")
print(f"Current: {df['sip_pct'].iloc[-1]:.1f}%")

print(f"\nPercentiles:")
for p in [5, 10, 25, 50, 75, 90, 95]:
    print(f"  {p}th: {df['sip_pct'].quantile(p/100):.1f}%")

In [ ]:
# How often is SIP below various thresholds?
print("\nFREQUENCY BELOW THRESHOLDS")
print("="*50)
for thresh in [40, 45, 50, 55, 60, 65, 70]:
    days_below = (df['sip_pct'] < thresh).sum()
    pct = days_below / len(df) * 100
    print(f"SIP < {thresh}%: {days_below} days ({pct:.1f}%)")

In [ ]:
# Visualize SIP with price
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.6, 0.4],
                    subplot_titles=['BTC Price', 'Supply in Profit %'])

fig.add_trace(go.Scatter(x=df.index, y=df['price'], name='Price'), row=1, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df['sip_pct'], name='SIP %',
                         line=dict(color='purple')), row=2, col=1)

# Add threshold lines
for thresh, color in [(50, 'red'), (60, 'orange'), (70, 'yellow')]:
    fig.add_hline(y=thresh, line_dash='dash', line_color=color, row=2, col=1)

fig.update_yaxes(type='log', row=1, col=1)
fig.update_layout(height=600, title_text='Supply in Profit % - Potential Buy Zones')
fig.show()

In [ ]:
# When was SIP < 50%?
low_sip = df[df['sip_pct'] < 50].copy()
print(f"\nPERIODS WITH SIP < 50%")
print("="*60)
print(f"Total days: {len(low_sip)}")

if len(low_sip) > 0:
    # Find distinct periods
    low_sip['gap'] = (low_sip.index.to_series().diff() > pd.Timedelta(days=7)).cumsum()
    periods = low_sip.groupby('gap').agg({
        'sip_pct': ['min', 'mean'],
        'price': ['first', 'last', 'min']
    })
    periods.columns = ['min_sip', 'avg_sip', 'start_price', 'end_price', 'min_price']

    # Add dates
    period_dates = low_sip.groupby('gap').apply(lambda x: (x.index.min(), x.index.max()))

    print(f"\nDistinct periods: {len(periods)}")
    print("\n" + "-"*80)
    for i, (idx, row) in enumerate(periods.iterrows()):
        start, end = period_dates.iloc[i]
        duration = (end - start).days + 1
        print(f"Period {i+1}: {start.date()} to {end.date()} ({duration} days)")
        print(f"  SIP: min={row['min_sip']:.1f}%, avg={row['avg_sip']:.1f}%")
        print(f"  Price: ${row['start_price']:,.0f} → ${row['end_price']:,.0f} (min: ${row['min_price']:,.0f})")
        print()
else:
    print("\n⚠️ SIP never dropped below 50% in this dataset!")
    print("This is actually rare - let's look at higher thresholds.")

---
## 2. Compare to SOPR Signal

In [ ]:
# How correlated are SIP and SOPR?
print("CORRELATION ANALYSIS")
print("="*50)
print(f"SIP vs SOPR: {df['sip_pct'].corr(df['sopr']):.3f}")
print(f"SIP vs STH SOPR: {df['sip_pct'].corr(df['sopr_sth']):.3f}")
print(f"SIP vs MVRV: {df['sip_pct'].corr(df['mvrv']):.3f}")

# When SOPR signals fire, what's the SIP?
sopr_signal = (df['sopr'] < 1) & (df['sopr_sth'] < 1)
print(f"\nWhen SOPR double cap fires:")
print(f"  Avg SIP: {df.loc[sopr_signal, 'sip_pct'].mean():.1f}%")
print(f"  Min SIP: {df.loc[sopr_signal, 'sip_pct'].min():.1f}%")
print(f"  Max SIP: {df.loc[sopr_signal, 'sip_pct'].max():.1f}%")

In [ ]:
# Visualize both signals
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, row_heights=[0.4, 0.3, 0.3],
                    subplot_titles=['BTC Price', 'Supply in Profit %', 'SOPR & STH SOPR'])

fig.add_trace(go.Scatter(x=df.index, y=df['price'], name='Price'), row=1, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df['sip_pct'], name='SIP %'), row=2, col=1)
fig.add_hline(y=50, line_dash='dash', line_color='red', row=2, col=1)
fig.add_hline(y=60, line_dash='dot', line_color='orange', row=2, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df['sopr'], name='SOPR'), row=3, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['sopr_sth'], name='STH SOPR'), row=3, col=1)
fig.add_hline(y=1, line_dash='dash', line_color='red', row=3, col=1)

fig.update_yaxes(type='log', row=1, col=1)
fig.update_layout(height=700, title_text='SIP vs SOPR Signals')
fig.show()

---
## 3. Test SIP as Entry Signal

In [ ]:
# Use same backtest framework as SOPR
df_test = df[df.index >= '2018-12-15'].copy()
close = df_test['price']

def create_entry_signal(df, threshold):
    """Entry when SIP drops below threshold (first day)."""
    below = df['sip_pct'] < threshold
    entries = below & ~below.shift(1).fillna(False)
    return entries

In [ ]:
def backtest_mvrv_trailing(
    df, entries,
    mvrv_trigger=2.25,
    trailing_pct=0.20,
    stop_loss=0.20,
    max_hold_days=365
):
    """Same exit strategy as our best SOPR strategy."""
    trades = []
    entry_indices = entries[entries].index.tolist()
    close = df['price']
    
    i = 0
    while i < len(entry_indices):
        entry_date = entry_indices[i]
        entry_idx = df.index.get_loc(entry_date)
        entry_price = close.iloc[entry_idx]
        
        peak_price = entry_price
        trailing_active = False
        
        exit_date = None
        exit_price = None
        exit_reason = None
        
        for j in range(entry_idx + 1, len(df)):
            current_date = df.index[j]
            current_price = close.iloc[j]
            current_mvrv = df['mvrv'].iloc[j]
            days_held = j - entry_idx
            
            if current_price > peak_price:
                peak_price = current_price
            
            pnl = (current_price - entry_price) / entry_price
            
            if not trailing_active and current_mvrv >= mvrv_trigger:
                trailing_active = True
            
            if trailing_active:
                trail_stop = peak_price * (1 - trailing_pct)
                if current_price <= trail_stop:
                    exit_date = current_date
                    exit_price = trail_stop
                    exit_reason = 'mvrv_trail'
                    break
            
            if not trailing_active and stop_loss and pnl <= -stop_loss:
                exit_date = current_date
                exit_price = entry_price * (1 - stop_loss)
                exit_reason = 'stop_loss'
                break
            
            if days_held >= max_hold_days:
                exit_date = current_date
                exit_price = current_price
                exit_reason = 'max_hold'
                break
        
        if exit_date is None:
            exit_date = df.index[-1]
            exit_price = close.iloc[-1]
            exit_reason = 'end_of_data'
        
        pnl = (exit_price - entry_price) / entry_price
        trades.append({
            'entry_date': entry_date,
            'exit_date': exit_date,
            'entry_price': entry_price,
            'exit_price': exit_price,
            'pnl_pct': pnl,
            'days_held': (exit_date - entry_date).days,
            'exit_reason': exit_reason,
            'entry_sip': df.loc[entry_date, 'sip_pct']
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_date:
            i += 1
    
    return pd.DataFrame(trades)

In [ ]:
# Test different SIP thresholds
thresholds = [40, 45, 50, 55, 60, 65, 70, 75, 80, 85]

print("SIP THRESHOLD COMPARISON (In-Sample)")
print("="*100)
print(f"{'Threshold':<12} {'Signals':>10} {'Trades':>10} {'Return':>12} {'Win Rate':>10} {'Avg Days':>10}")
print("-"*100)

sip_results = []

for thresh in thresholds:
    entries = create_entry_signal(df_test, thresh)
    n_signals = entries.sum()
    
    if n_signals == 0:
        print(f"SIP < {thresh}%{'':<4} {n_signals:>10} {'-':>10} {'-':>12} {'-':>10} {'-':>10}")
        continue
    
    trades = backtest_mvrv_trailing(df_test, entries)
    
    total_return = (1 + trades['pnl_pct']).prod() - 1 if len(trades) > 0 else 0
    win_rate = (trades['pnl_pct'] > 0).mean() if len(trades) > 0 else 0
    avg_days = trades['days_held'].mean() if len(trades) > 0 else 0
    
    print(f"SIP < {thresh}%{'':<4} {n_signals:>10} {len(trades):>10} {total_return*100:>11.0f}% "
          f"{win_rate*100:>9.0f}% {avg_days:>10.0f}")
    
    sip_results.append({
        'threshold': thresh,
        'signals': n_signals,
        'trades': len(trades),
        'total_return': total_return,
        'win_rate': win_rate,
        'trades_df': trades
    })

---
## 4. Walk-Forward Validation

In [ ]:
def walk_forward(df, threshold, mvrv_trigger=2.25, trailing_pct=0.20):
    """Walk-forward validation."""
    results = []
    close = df['price']
    
    train_days = 365
    test_days = 90
    step_days = 90
    
    total_days = len(df)
    n_folds = (total_days - train_days) // step_days
    
    for fold in range(n_folds):
        test_start = train_days + fold * step_days
        test_end = min(test_start + test_days, total_days)
        
        test_df = df.iloc[test_start:test_end]
        test_close = close.iloc[test_start:test_end]
        
        entries = create_entry_signal(test_df, threshold)
        trades = backtest_mvrv_trailing(test_df, entries, mvrv_trigger, trailing_pct)
        
        strat_return = (1 + trades['pnl_pct']).prod() - 1 if len(trades) > 0 else 0
        hold_return = (test_close.iloc[-1] / test_close.iloc[0]) - 1
        
        results.append({
            'strat_return': strat_return,
            'hold_return': hold_return,
            'beat_hold': strat_return > hold_return,
            'n_trades': len(trades)
        })
    
    wf_df = pd.DataFrame(results)
    return wf_df['beat_hold'].mean(), (wf_df['strat_return'] - wf_df['hold_return']).mean(), wf_df['n_trades'].sum()

In [ ]:
# Walk-forward for each threshold
print("\nWALK-FORWARD VALIDATION")
print("="*80)
print(f"{'Threshold':<12} {'Beat Rate':>15} {'Avg Excess':>15} {'Total Trades':>15}")
print("-"*80)

wf_results = []

for thresh in thresholds:
    beat_rate, avg_excess, total_trades = walk_forward(df_test, thresh)
    
    print(f"SIP < {thresh}%{'':<4} {beat_rate*100:>14.0f}% {avg_excess*100:>+14.1f}% {total_trades:>15}")
    
    wf_results.append({
        'threshold': thresh,
        'beat_rate': beat_rate,
        'avg_excess': avg_excess,
        'total_trades': total_trades
    })

wf_df = pd.DataFrame(wf_results)

In [ ]:
# Compare to SOPR baseline
print("\n\nCOMPARISON TO SOPR BASELINE")
print("="*60)
print(f"SOPR Double Cap + MVRV Trail: 62% beat rate")

if len(wf_df[wf_df['total_trades'] > 0]) > 0:
    best_idx = wf_df[wf_df['total_trades'] > 0]['beat_rate'].idxmax()
    print(f"\nBest SIP threshold: SIP < {wf_df.loc[best_idx, 'threshold']}%")
    print(f"Best SIP beat rate: {wf_df.loc[best_idx, 'beat_rate']*100:.0f}%")
else:
    print("\n⚠️ No SIP thresholds produced enough signals for comparison")

In [ ]:
# Visualize
valid_wf = wf_df[wf_df['total_trades'] > 0]

if len(valid_wf) > 0:
    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=[f"SIP < {t}%" for t in valid_wf['threshold']],
        y=valid_wf['beat_rate'] * 100,
        marker_color=['green' if x > 0.62 else 'orange' if x > 0.54 else 'gray' for x in valid_wf['beat_rate']],
        text=[f"{x:.0f}%" for x in valid_wf['beat_rate']*100],
        textposition='outside'
    ))

    fig.add_hline(y=54, line_dash='dash', line_color='orange', annotation_text='SOPR baseline 54%')
    fig.add_hline(y=62, line_dash='dash', line_color='green', annotation_text='SOPR+MVRV 62%')

    fig.update_layout(
        title='Walk-Forward Beat Rate by SIP Threshold',
        yaxis_title='Beat Rate %',
        height=500
    )
    fig.show()
else:
    print("Not enough data to visualize")

---
## 5. Combine SIP with SOPR?

In [ ]:
# What if we require BOTH signals?
# SOPR < 1 AND STH SOPR < 1 AND SIP < threshold

def combined_entry(df, sip_threshold):
    """Entry when SOPR double cap AND SIP below threshold."""
    sopr_signal = (df['sopr'] < 1) & (df['sopr_sth'] < 1)
    sip_signal = df['sip_pct'] < sip_threshold
    combined = sopr_signal & sip_signal
    entries = combined & ~combined.shift(1).fillna(False)
    return entries

print("COMBINED SIGNAL: SOPR + SIP")
print("="*80)
print(f"{'SIP Threshold':<15} {'Signals':>10} {'Beat Rate':>15} {'Avg Excess':>15}")
print("-"*80)

combined_results = []

for sip_thresh in [50, 55, 60, 65, 70, 75, 80, 85, 90]:
    # Walk-forward
    results = []
    close = df_test['price']
    
    train_days = 365
    test_days = 90
    step_days = 90
    
    total_days = len(df_test)
    n_folds = (total_days - train_days) // step_days
    total_signals = 0
    
    for fold in range(n_folds):
        test_start = train_days + fold * step_days
        test_end = min(test_start + test_days, total_days)
        
        test_df = df_test.iloc[test_start:test_end]
        test_close = close.iloc[test_start:test_end]
        
        entries = combined_entry(test_df, sip_thresh)
        total_signals += entries.sum()
        trades = backtest_mvrv_trailing(test_df, entries)
        
        strat_return = (1 + trades['pnl_pct']).prod() - 1 if len(trades) > 0 else 0
        hold_return = (test_close.iloc[-1] / test_close.iloc[0]) - 1
        
        results.append({
            'strat_return': strat_return,
            'hold_return': hold_return,
            'beat_hold': strat_return > hold_return
        })
    
    wf_result = pd.DataFrame(results)
    beat_rate = wf_result['beat_hold'].mean()
    avg_excess = (wf_result['strat_return'] - wf_result['hold_return']).mean()
    
    print(f"SOPR + SIP<{sip_thresh}%{'':<3} {total_signals:>10} {beat_rate*100:>14.0f}% {avg_excess*100:>+14.1f}%")
    
    combined_results.append({
        'sip_threshold': sip_thresh,
        'signals': total_signals,
        'beat_rate': beat_rate,
        'avg_excess': avg_excess
    })

---
## 6. Summary

In [ ]:
print("\n" + "="*80)
print("SUPPLY IN PROFIT ANALYSIS SUMMARY")
print("="*80)

# Best standalone SIP
valid_wf = wf_df[wf_df['total_trades'] > 0]
if len(valid_wf) > 0:
    best_sip = valid_wf.loc[valid_wf['beat_rate'].idxmax()]
    print(f"\n📊 STANDALONE SIP SIGNAL")
    print(f"   Best threshold: SIP < {best_sip['threshold']}%")
    print(f"   Beat rate: {best_sip['beat_rate']*100:.0f}%")
    print(f"   Avg excess: {best_sip['avg_excess']*100:+.1f}%")
else:
    print(f"\n📊 STANDALONE SIP SIGNAL")
    print(f"   ⚠️ Not enough signals to test standalone")
    best_sip = None

# Best combined
combined_df = pd.DataFrame(combined_results)
valid_combined = combined_df[combined_df['signals'] > 0]
if len(valid_combined) > 0:
    best_combined = valid_combined.loc[valid_combined['beat_rate'].idxmax()]
    print(f"\n📊 COMBINED SIGNAL (SOPR + SIP)")
    print(f"   Best threshold: SOPR double cap + SIP < {best_combined['sip_threshold']}%")
    print(f"   Beat rate: {best_combined['beat_rate']*100:.0f}%")
    print(f"   Avg excess: {best_combined['avg_excess']*100:+.1f}%")
else:
    print(f"\n📊 COMBINED SIGNAL (SOPR + SIP)")
    print(f"   ⚠️ Not enough signals to test combined")
    best_combined = None

# Compare to SOPR baseline
print(f"\n📊 COMPARISON")
print(f"   SOPR alone:     62% beat rate")
if best_sip is not None:
    print(f"   SIP alone:      {best_sip['beat_rate']*100:.0f}% beat rate")
if best_combined is not None:
    print(f"   SOPR + SIP:     {best_combined['beat_rate']*100:.0f}% beat rate")

# Verdict
print(f"\n🎯 VERDICT:")
if best_sip is not None and best_sip['beat_rate'] > 0.62:
    print(f"   ✅ SIP alone BEATS SOPR! Consider as alternative signal.")
elif best_combined is not None and best_combined['beat_rate'] > 0.62:
    print(f"   ✅ Combined signal BEATS SOPR alone! Use as enhancement.")
else:
    print(f"   ⚠️ SIP doesn't improve on SOPR. Stick with SOPR + MVRV trail.")

print("\n" + "="*80)

In [ ]:
# Save results
import json

# Convert numpy types to native Python
def to_native(obj):
    if isinstance(obj, dict):
        return {k: to_native(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [to_native(v) for v in obj]
    elif hasattr(obj, 'item'):
        return obj.item()
    return obj

results = {
    'signal': 'supply_in_profit',
    'standalone_results': to_native(wf_df.to_dict('records')),
    'combined_results': to_native(combined_results),
    'best_standalone': {
        'threshold': int(best_sip['threshold']) if best_sip is not None else None,
        'beat_rate': float(best_sip['beat_rate']) if best_sip is not None else None,
        'avg_excess': float(best_sip['avg_excess']) if best_sip is not None else None
    },
    'best_combined': {
        'sip_threshold': int(best_combined['sip_threshold']) if best_combined is not None else None,
        'beat_rate': float(best_combined['beat_rate']) if best_combined is not None else None,
        'avg_excess': float(best_combined['avg_excess']) if best_combined is not None else None
    },
    'baseline_sopr': {'beat_rate': 0.62}
}

with open('../data/supply_in_profit_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Saved to ../data/supply_in_profit_results.json")